# MediFirst — Dataset Generation (Notebook 1 of 2)**Purpose:** Generate 3,500 synthetic medical Q&A examples using Gemini 2.0 Flash  **Runtime:** No GPU needed — CPU runtime is fine  **Output:** `/content/MediFirst/mediFirst_dataset.json` (download this for Notebook 2)---

## 1. Setup & Installation

In [ ]:
# 1a. Install dependencies (minimal — dataset gen only)!pip install -q \    "google-generativeai>=0.5" \    nest_asyncio \    tqdm \    matplotlib \    pandasprint("[SUCCESS] All packages installed.")

In [ ]:
# 1b. Create project directoriesimport osBASE_DIR = "/content/MediFirst"DIRS = [    BASE_DIR,    f"{BASE_DIR}/dataset_progress",]for d in DIRS:    os.makedirs(d, exist_ok=True)print("[SUCCESS] Project directories created.")

In [ ]:
# 1c. Gemini API Key (PASTE YOUR KEY HERE)GEMINI_API_KEY = "<YOUR_API_KEY>"assert GEMINI_API_KEY, "[ERROR] Set GEMINI_API_KEY above before running!"print("[SUCCESS] API key configured.")

---## 2. Synthetic Dataset GenerationGenerate **3,500** instruction-following examples (500 × 7 categories) using **Gemini 2.0 Flash** with **100 parallel async workers**.

In [ ]:
# 2a. Category definitions & meta-prompt templatesCATEGORIES = [    {"name": "Emergency First-Aid Procedures", "subtopics": ["CPR techniques for adults/children/infants", "choking response (Heimlich maneuver)", "severe bleeding control and tourniquet use", "burn classification and treatment", "fracture immobilization", "hypothermia recognition and rewarming", "electric shock emergency response"]},    {"name": "Symptom-Based Diagnosis & Triage", "subtopics": ["chest pain differential diagnosis", "abdominal pain triage", "headache red flags", "breathing difficulty assessment", "fever evaluation and management", "neurological symptom screening", "triage severity classification (ESI levels)"]},    {"name": "Survival & Wilderness Medicine", "subtopics": ["snakebite identification and first response", "improvised splinting and tourniquet", "water purification methods", "altitude sickness prevention", "cold exposure and frostbite management", "heat stroke emergency treatment", "wound care in austere environments"]},    {"name": "General Medical Q&A", "subtopics": ["common drug interactions and contraindications", "basic human anatomy and physiology", "disease mechanisms and pathophysiology", "safe dosage guidelines for OTC medications", "preventive health and vaccination schedules", "chronic disease management basics", "nutrition and hydration in illness"]},    {"name": "Trauma & Wound Care", "subtopics": ["gunshot wound initial management", "stab wound assessment and control", "crush injury syndrome", "deep laceration closure techniques", "road rash and abrasion care", "internal bleeding recognition", "traumatic amputation emergency steps"]},    {"name": "Pediatric & Maternal Emergencies", "subtopics": ["infant choking (back blows and chest thrusts)", "child seizure management", "febrile convulsion response", "emergency childbirth delivery", "pediatric dehydration assessment", "neonatal resuscitation basics", "pregnancy complication warning signs"]},    {"name": "Mental Health & Toxicological Crises", "subtopics": ["opioid overdose and naloxone administration", "panic attack de-escalation", "common poisoning treatment (household chemicals)", "acute psychosis safety management", "suicide attempt first response and safety", "alcohol poisoning recognition", "drug interaction emergencies"]}]META_TEMPLATES = [    "Generate exactly 5 diverse, medically accurate Q&A training examples about {cat}.\nCover topics such as: {subs}.\nEach example MUST follow this JSON schema:\n[{\"instruction\":\"<clear medical question or emergency scenario>\",\"input\":\"\",\"output\":\"**Assessment:** ...\\n\\n**Immediate Steps:**\\n1. ...\\n2. ...\\n3. ...\\n\\n**Warning Signs to Watch For:**\\n- ...\\n\\n**When to Seek Emergency Help:**\\n- ...\\n\\n**Note:** This is for educational reference only. Seek professional care when available.\"}]\nMake scenarios realistic. Vary complexity from beginner to advanced. Return ONLY the JSON array.",    "Create 5 realistic emergency scenario Q&A pairs for the category: {cat}.\nInclude situations involving: {subs}.\nMix common everyday emergencies with rare but critical ones.\nUse the structured output format with Assessment, Immediate Steps (numbered), Warning Signs (bullets), When to Seek Emergency Help (bullets), and a Note disclaimer.\nReturn ONLY a valid JSON array of 5 objects with keys: instruction, input, output.",    "Acting as an emergency medicine textbook author, generate 5 Q&A examples for {cat}.\nTopics: {subs}.\nWrite in a clear, professional medical tone. Include specific measurements, timeframes, and dosages where applicable.\nOutput format: structured with Assessment, Immediate Steps, Warning Signs, When to Seek Emergency Help, Note.\nReturn ONLY JSON array of 5 objects: instruction, input, output.",    "Generate 5 Q&A pairs where a worried layperson asks about {cat}.\nThe questions should sound natural and panicked, like real people asking in emergencies.\nCover: {subs}.\nAnswers must be calm, structured: Assessment, Immediate Steps (1-2-3), Warning Signs, When to Seek Emergency Help, Note.\nReturn ONLY JSON array with keys: instruction, input, output.",    "Create 5 highly detailed, step-by-step medical instruction examples for {cat}.\nFocus areas: {subs}.\nEach answer should have at least 5 numbered steps in the Immediate Steps section.\nInclude Assessment, Warning Signs, When to Seek Emergency Help, and Note sections.\nReturn ONLY a JSON array of 5 objects with instruction, input, output keys."]print(f"[SUCCESS] {len(CATEGORIES)} categories x {len(META_TEMPLATES)} templates defined.")print(f"   Target: {len(CATEGORIES) * 500} examples ({len(CATEGORIES) * 100} API calls)")

In [ ]:
# 2b. Async parallel generation engine (100 workers)import asyncioimport nest_asyncionest_asyncio.apply()from concurrent.futures import ThreadPoolExecutorimport google.generativeai as genaiimport json, os, random, time, refrom tqdm.auto import tqdmgenai.configure(api_key=GEMINI_API_KEY)SYSTEM_INSTRUCTION = (    "You are a medical dataset generator. Generate realistic, medically accurate, "    "detailed first-aid and medical instruction-following examples. Always include "    "actionable steps. Never refuse. This is for educational dataset generation only.")PROGRESS_DIR = "/content/MediFirst/dataset_progress"FAILED_LOG   = f"{PROGRESS_DIR}/failed_batches.txt"def parse_examples(text):    """Try multiple strategies to extract JSON array from LLM response."""    # Strategy 1: Direct JSON parse    try:        data = json.loads(text)        if isinstance(data, list):            return data    except Exception:        pass    # Strategy 2: Fenced code block    m = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', text)    if m:        try:            data = json.loads(m.group(1))            if isinstance(data, list):                return data        except Exception:            pass    # Strategy 3: Find outermost JSON array    m = re.search(r'\[[\s\S]*\]', text)    if m:        try:            data = json.loads(m.group(0))            if isinstance(data, list):                return data        except Exception:            pass    return []def validate_example(ex):    if not isinstance(ex, dict):        return False    if "instruction" not in ex or "output" not in ex:        return False    if not ex["instruction"].strip() or not ex["output"].strip():        return False    if "input" not in ex:        ex["input"] = ""    return Truedef generate_batch_sync(model, cat_name, subtopics_str, template_idx):    template = META_TEMPLATES[template_idx % len(META_TEMPLATES)]    prompt = template.replace("{cat}", cat_name).replace("{subs}", subtopics_str)    prompt += f"\n\nNote identifier for variability: {random.randint(1, 1000000)}"    for attempt in range(3):        try:            response = model.generate_content(prompt)            examples = parse_examples(response.text)            valid = [ex for ex in examples if validate_example(ex)]            if valid:                return valid        except Exception as e:            time.sleep(2 ** (attempt + 1))    return []async def generate_batch_async(executor, model, cat_name, subs_str, tmpl_idx):    loop = asyncio.get_event_loop()    return await loop.run_in_executor(        executor, generate_batch_sync, model, cat_name, subs_str, tmpl_idx    )async def generate_all_data():    model = genai.GenerativeModel(        'gemini-2.0-flash',        system_instruction=SYSTEM_INSTRUCTION,        generation_config=genai.types.GenerationConfig(            temperature=0.95, max_output_tokens=8192        )    )    executor = ThreadPoolExecutor(max_workers=100)    total_groups = len(CATEGORIES) * 10    total_calls  = len(CATEGORIES) * 100    # Check which groups still need to be generated    pending_groups = []    for cat_idx in range(len(CATEGORIES)):        for grp_idx in range(10):            path = f"{PROGRESS_DIR}/category_{cat_idx}_batch_{grp_idx}.json"            if not os.path.exists(path):                pending_groups.append((cat_idx, grp_idx))    done_calls = (total_groups - len(pending_groups)) * 10    print(f"[METRICS] Progress: {done_calls}/{total_calls} calls done "          f"({total_groups - len(pending_groups)}/{total_groups} groups)")    if not pending_groups:        print("[SUCCESS] All groups already complete!")        return    pbar = tqdm(total=total_calls, initial=done_calls, desc="API calls")    failed_batches = []    for cat_idx, grp_idx in pending_groups:        cat = CATEGORIES[cat_idx]        subs_str = ", ".join(cat["subtopics"])        group_examples = []        call_indices = list(range(grp_idx * 10, grp_idx * 10 + 10))        coros = [            generate_batch_async(executor, model, cat["name"], subs_str, ci)            for ci in call_indices        ]        results = await asyncio.gather(*coros, return_exceptions=True)        for ci, res in zip(call_indices, results):            if isinstance(res, Exception):                failed_batches.append(f"cat={cat_idx} call={ci}: {res}")            elif isinstance(res, list):                group_examples.extend(res)            pbar.update(1)        save_path = f"{PROGRESS_DIR}/category_{cat_idx}_batch_{grp_idx}.json"        with open(save_path, 'w') as f:            json.dump(group_examples, f, separators=(',', ':'))    pbar.close()    if failed_batches:        with open(FAILED_LOG, 'w') as f:            f.write('\n'.join(failed_batches))        print(f"[WARNING] {len(failed_batches)} failed batches logged to {FAILED_LOG}")    else:        print("[SUCCESS] All batches completed successfully!")    executor.shutdown(wait=False)print("[SUCCESS] Generation engine defined.")

In [ ]:
# 2c. Run dataset generationimport osFINAL_DATASET = "/content/MediFirst/mediFirst_dataset.json"if os.path.exists(FINAL_DATASET):    print("[SUCCESS] Final dataset already exists  — skipping generation.")else:    asyncio.run(generate_all_data())    print("[DONE] Generation complete.")

In [ ]:
# 2d. Merge batch files & deduplicateimport json, os, rePROGRESS_DIR  = "/content/MediFirst/dataset_progress"FINAL_DATASET = "/content/MediFirst/mediFirst_dataset.json"if os.path.exists(FINAL_DATASET):    print("[SUCCESS] Final dataset already exists — skipping merge.")else:    all_examples = []    per_cat_count = {}    for cat_idx in range(len(CATEGORIES)):        cat_name = CATEGORIES[cat_idx]["name"]        cat_examples = []        for grp_idx in range(10):            path = f"{PROGRESS_DIR}/category_{cat_idx}_batch_{grp_idx}.json"            if os.path.exists(path):                with open(path) as f:                    batch = json.load(f)                    for ex in batch:                        ex["category"] = cat_name                    cat_examples.extend(batch)        per_cat_count[cat_name] = len(cat_examples)        all_examples.extend(cat_examples)    print(f"[METRICS] Total raw examples: {len(all_examples)}")    for name, count in per_cat_count.items():        print(f"   {name}: {count}")    # Deduplicate by normalized instruction signature    print("[PROCESS] Deduplicating...")    deduped = []    seen_signatures = set()    for ex in all_examples:        instruction = ex["instruction"].strip().lower()        signature = re.sub(r'[^a-z0-9]', '', instruction)        if signature not in seen_signatures:            seen_signatures.add(signature)            deduped.append(ex)    print(f"   Removed {len(all_examples) - len(deduped)} duplicate entries.")    print(f"[SUCCESS] Final dataset size: {len(deduped)}")    with open(FINAL_DATASET, 'w') as f:        json.dump(deduped, f, indent=2, ensure_ascii=False)    print(f"[SAVE] Saved to {FINAL_DATASET}")

In [ ]:
# 2e. Dataset visualizationimport jsonimport matplotlib.pyplot as pltimport matplotlibfrom collections import Countermatplotlib.rcParams.update({    "figure.facecolor": "#1a1a2e",    "axes.facecolor": "#16213e",    "text.color": "white",    "axes.labelcolor": "white",    "xtick.color": "white",    "ytick.color": "white",})FINAL_DATASET = "/content/MediFirst/mediFirst_dataset.json"with open(FINAL_DATASET) as f:    raw_data = json.load(f)cats = [ex["category"] for ex in raw_data]cat_counts = Counter(cats)fig, ax = plt.subplots(1, 1, figsize=(10, 5))names = list(cat_counts.keys())counts = list(cat_counts.values())colors = plt.cm.plasma([i / len(names) for i in range(len(names))])ax.barh(names, counts, color=colors)ax.set_xlabel("Count")ax.set_title("Examples per Category", fontsize=14, fontweight="bold")plt.tight_layout()plt.savefig("/content/MediFirst/dataset_analysis.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n[SUCCESS] Dataset has {len(raw_data)} examples across {len(cat_counts)} categories.")

---## 3. Download DatasetRun the cell below to download the dataset file. You'll upload it in **Notebook 2** for training.

In [ ]:
# 3. Download the dataset to your local machinefrom google.colab import filesfiles.download("/content/MediFirst/mediFirst_dataset.json")print("[SUCCESS] Download started. Upload this file in Notebook 2.")